# Health Data Pipeline: Branch Overview

This notebook walks through the work in `feat/dashboard-prototype`: source data, canonical ingestion, daily features, embeddings, forecasts, and the React dashboard. It reads existing local artifacts when present and reports aggregate metadata only; it does not print raw health observations or rerun the pipeline.

**Interpretation guardrails:** activity and recovery scores are relative to each subject, not clinical measures. UMAP and t-SNE are exploratory projections, not held-out model evaluation. The React dashboard consumes static JSON exported from the generated daily-feature artifact; its what-if controls simulate scenarios on top of that real subject data.

In [11]:
from collections import Counter
from pathlib import Path
import json

import altair as alt
import polars as pl
from IPython.display import display

# Support execution with the kernel rooted either at the repository or notebooks/.
_candidates = [Path.cwd(), Path.cwd().parent]
ROOT = next((path.resolve() for path in _candidates if (path / "pyproject.toml").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Could not locate repository root (pyproject.toml).")
DATA = ROOT / "data"
MODEL = DATA / "model"
PROCESSED = DATA / "processed"
alt.data_transformers.disable_max_rows()

stage_data = [
    {"stage": "01 · Ingest", "label": "XML · CSV · JSON · PMData", "kind": "Data"},
    {"stage": "02 · Normalize", "label": "Schema · units · validation", "kind": "Engineering"},
    {"stage": "03 · Features", "label": "Daily signals · axes", "kind": "Features"},
    {"stage": "04 · Embed", "label": "PCA · autoencoder · projections", "kind": "Models"},
    {"stage": "05 · Forecast", "label": "Leakage-safe baselines", "kind": "Models"},
    {"stage": "06 · Visualize", "label": "Marimo analysis · React dashboard", "kind": "Product"},
]
flow = alt.Chart(alt.Data(values=stage_data)).mark_rect(cornerRadius=4).encode(
    x=alt.X("stage:O", sort=[row["stage"] for row in stage_data], axis=None),
    y=alt.value(20),
    color=alt.Color("kind:N", legend=None),
    tooltip=["label:N", "kind:N"],
)
flow_labels = alt.Chart(alt.Data(values=stage_data)).mark_text(
    color="white", fontSize=11, lineBreak="\n", lineHeight=14
).encode(
    x=alt.X("stage:O", sort=[row["stage"] for row in stage_data], axis=None),
    y=alt.value(43),
    text="label:N",
)
display((flow + flow_labels).properties(width=850, height=80, title="From raw exports to an interactive concept"))

processed_files = sorted(PROCESSED.glob("*.parquet")) if PROCESSED.exists() else []
format_counts = Counter(path.stem.split("_", 2)[-1] for path in processed_files)
source_inventory = pl.DataFrame(
    [{"source_format": name, "processed_files": count} for name, count in sorted(format_counts.items())]
) if format_counts else pl.DataFrame(schema={"source_format": pl.String, "processed_files": pl.Int64})
print(f"Repository: {ROOT}")
print(f"Processed subject files: {len(processed_files)}")
display(source_inventory)

alt.LayerChart(...)

Repository: C:\Users\itzya\Documents\wpi-datathon-2026
Processed subject files: 20


source_format,processed_files
str,i64
"""apple_xml""",2
"""flat_csv""",1
"""nested_json""",1
"""pmdata""",16


## 1. Getting the data

The ingestion package accepts four source families: Apple Health XML (`data/ours/`), flat CSV and nested JSON exports (`data/other/`), and participant-folder PMData (`data/pmdata/`). Raw data, subject mappings, and parsed files are local and gitignored. Start with `uv run health-prep scan`, fill in any local subject/source mappings it requests, then run `uv run health-prep parse`.

This workspace currently has **20 processed subject files** (2 Apple XML, 1 CSV, 1 nested JSON, and 16 PMData). The figures below document why the source formats need distinct readers and how much their metric coverage differs; coverage is not uniform across people or days.

![Overview of the nested health export format](../figures/kaggle_json_overview.png)

![Comparison of dataset richness across source collections](../figures/dataset_richness.png)

## 2. Data engineering

`health-prep` dispatches each input to a format-specific reader, maps source names to broad source types, converts supported measurements into canonical units, then validates the shared observation schema before writing Parquet. The output keeps both UTC and local timestamps: daily features use local calendar dates, while identifiers are anonymized subject IDs. The metric catalogue provides canonical names and domain/daily aggregation metadata.

The shared row schema is `subject_id`, `source_format`, `metric`, `unit`, `value_num` or `value_str`, timestamps, and `source_type`. Values remain at observation grain here; daily aggregation happens in the next stage. See [README.md](../README.md), [pipeline.py](../src/health/pipeline.py), and the format readers under `src/health/readers/`.

## 3. Feature engineering

In [9]:
daily_path = MODEL / "daily_features.parquet"
if daily_path.exists():
    daily = pl.read_parquet(daily_path)
    core = [name for name in ("steps", "sleep_hours", "hr_mean_bpm") if name in daily.columns]
    coverage_rows = [
        {"feature": name, "coverage": daily[name].is_not_null().mean()}
        for name in core
    ]
    print(
        f"Daily feature artifact: {daily.height:,} subject-days · "
        f"{daily['subject_id'].n_unique()} subjects · {len(daily.columns)} columns"
    )
    if coverage_rows:
        coverage_chart = alt.Chart(alt.Data(values=coverage_rows)).mark_bar(cornerRadiusEnd=3).encode(
            x=alt.X("coverage:Q", scale=alt.Scale(domain=[0, 1]), axis=alt.Axis(format="%"), title="Observed days"),
            y=alt.Y("feature:N", sort="-x", title=None),
            color=alt.Color("feature:N", legend=None),
            tooltip=[alt.Tooltip("feature:N"), alt.Tooltip("coverage:Q", format=".1%")],
        ).properties(title="Core signal availability in the selected daily table", height=120)
        display(coverage_chart)

    score_columns = [name for name in ("activity_score", "recovery_score") if name in daily.columns]
    if score_columns:
        score_rows = (
            daily.group_by(pl.col("date").dt.truncate("1mo").alias("month"))
            .agg(*(pl.col(name).mean().alias(name) for name in score_columns))
            .sort("month")
            .unpivot(index="month", on=score_columns, variable_name="axis", value_name="mean_score")
            .drop_nulls("mean_score")
            .with_columns(pl.col("month").dt.strftime("%Y-%m-%d"))
            .to_dicts()
        )
        score_chart = alt.Chart(alt.Data(values=score_rows)).mark_line(point=True).encode(
            x=alt.X("month:T", title="Month"),
            y=alt.Y("mean_score:Q", title="Mean relative-to-self score"),
            color=alt.Color("axis:N", title="Composite axis"),
            tooltip=["month:T", "axis:N", alt.Tooltip("mean_score:Q", format=".2f")],
        ).properties(title="Monthly mean activity and recovery axes across subjects", height=250)
        display(score_chart)
else:
    daily = None
    print(f"No daily feature artifact found at {daily_path}. Run `uv run health-model` to create it.")

Daily feature artifact: 4,868 subject-days · 20 subjects · 50 columns


alt.Chart(...)

alt.Chart(...)

Daily features turn observation rows into one row per subject and local day. Cumulative measurements are summed, sampled levels are averaged, and sleep is assigned to a noon-to-noon day so an overnight session stays together. Missing sensor readings remain null rather than becoming zeros.

The feature set adds heart-rate distribution/spectral summaries, causal 7- and 28-day trends, and two readable composites. **Activity** and **recovery** average direction-aligned robust z-scores relative to each subject; their values are not cross-person rankings. Feature-rich windows are selected using completeness on the common core signals. See [features.py](../src/health/features.py) and [embedding-baseline.md](../docs/embedding-baseline.md).

## 4. Embeddings

PCA and a small denoising autoencoder are fit on earlier days and compared by reconstruction error on observed cells in each subject's latest 20%. Lower standardized RMSE means more accurate reconstruction for this benchmark. UMAP and t-SNE are included as **exploratory** projections for cluster inspection only; they are fit for visualization and should not be read as held-out performance or preserve global distances.

In [10]:
embedding_path = MODEL / "embeddings.parquet"
evaluation_path = MODEL / "evaluation.json"
if embedding_path.exists() and evaluation_path.exists():
    embeddings = pl.read_parquet(embedding_path).with_columns(pl.col("subject_id").cast(pl.String))
    embedding_report = json.loads(evaluation_path.read_text())
    print(
        f"Embedding artifact: {embeddings.height:,} subject-days · "
        f"{embeddings['subject_id'].n_unique()} subjects · "
        f"{len(embedding_report.get('features', []))} model features"
    )

    coordinates = [name for name in ("pc_1", "pc_2") if name in embeddings.columns]
    if len(coordinates) == 2:
        selected_subject = embeddings["subject_id"].unique().sort().to_list()[0]
        trajectory = (
            embeddings.filter(pl.col("subject_id") == selected_subject)
            .drop_nulls(coordinates)
            .sort("date")
            .tail(180)
            .select("date", *coordinates)
            .with_columns(pl.col("date").cast(pl.String))
            .to_dicts()
        )
        trajectory_chart = alt.Chart(alt.Data(values=trajectory)).mark_line(point=True).encode(
            x=alt.X("pc_1:Q", title="PCA component 1"),
            y=alt.Y("pc_2:Q", title="PCA component 2"),
            order="date:T",
            color=alt.Color("date:T", title="Date", scale=alt.Scale(scheme="viridis")),
            tooltip=["date:T", alt.Tooltip("pc_1:Q", format=".2f"), alt.Tooltip("pc_2:Q", format=".2f")],
        ).properties(
            title=f"Recent 180-day PCA path · {selected_subject} (anonymized)",
            width=500,
            height=320,
        )
        display(trajectory_chart)

    reconstruction_rows = [
        {"model": model, "rmse": embedding_report[model]["rmse_standardized"]}
        for model in ("pca", "autoencoder")
        if model in embedding_report and "rmse_standardized" in embedding_report[model]
    ]
    if reconstruction_rows:
        display(pl.DataFrame(reconstruction_rows))
        reconstruction_chart = alt.Chart(alt.Data(values=reconstruction_rows)).mark_bar().encode(
            x=alt.X("rmse:Q", title="Held-out standardized RMSE (lower is better)"),
            y=alt.Y("model:N", sort="x", title=None),
            color=alt.Color("model:N", legend=None),
            tooltip=["model:N", alt.Tooltip("rmse:Q", format=".3f")],
        ).properties(title="Observed-cell reconstruction benchmark", height=100)
        display(reconstruction_chart)
else:
    embeddings = pl.DataFrame()
    embedding_report = {}
    print(f"Embedding artifacts missing under {MODEL}. Run `uv run health-model` first.")

Embedding artifact: 4,868 subject-days · 20 subjects · 24 model features


alt.Chart(...)

model,rmse
str,f64
"""pca""",1.01301
"""autoencoder""",0.826402


alt.Chart(...)

The current local embedding report covers 20 subjects and 4,868 days across 24 features. In this run, the autoencoder has lower held-out standardized RMSE than PCA; that is a benchmark result for this dataset, not enough by itself to establish a better or more general model. The path plot shows how one subject moves in the first two PCA dimensions over time; axes are learned and should not be interpreted as direct health measures.

## 5. Forecasting

Forecasts target the next activity and recovery scores at 1-, 3-, 7-, and 14-day horizons. Before fitting, stationarity checks choose per-subject/per-feature differencing orders using training history only; predictions are reconstructed to the score scale using origin-time anchors. Temporal splits keep future labels out of training. Six baselines/models are compared: persistence, weekly seasonal persistence, rolling mean, Ridge, LightGBM, and ElasticNet.

The pairwise statistics compare **per-subject MAE** with paired two-sided Wilcoxon signed-rank tests and Holm correction. A small adjusted p-value indicates evidence of a difference, not the probability one model is better. The observed lowest-MAE model can differ from a statistically supported winner; when comparisons are inconclusive the supported-winner field is null. See [forecast_cli.py](../src/health/forecast_cli.py) and [forecast.py](../src/health/forecast.py).

In [5]:
forecast_path = MODEL / "forecast_evaluation.json"
if forecast_path.exists():
    forecast_report = json.loads(forecast_path.read_text())
    forecast_rows = []
    winner_rows = []
    pairwise_rows = []
    for horizon, horizon_report in forecast_report.get("horizons", {}).items():
        for target, target_report in horizon_report.get("targets", {}).items():
            comparison = target_report.get("statistical_comparison", {})
            winner_rows.append(
                {
                    "horizon_days": int(horizon),
                    "target": target,
                    "empirical_best": comparison.get("empirical_best_model"),
                    "supported_best": comparison.get("statistically_supported_best_model"),
                    "subjects": comparison.get("n_subjects"),
                }
            )
            for model, metrics in target_report.items():
                if model != "statistical_comparison" and isinstance(metrics, dict) and "mae" in metrics:
                    forecast_rows.append(
                        {
                            "horizon_days": int(horizon),
                            "target": target,
                            "model": model,
                            "mae": metrics["mae"],
                        }
                    )
            if int(horizon) == 1:
                pairwise_rows.extend(
                    {"target": target, **item}
                    for item in comparison.get("pairwise_comparisons", [])
                )

    print(f"Forecast horizons in report: {forecast_report.get('horizon_days', [])}")
    if forecast_rows:
        forecast_chart = alt.Chart(alt.Data(values=forecast_rows)).mark_line(point=True).encode(
            x=alt.X("horizon_days:O", sort=sorted({row["horizon_days"] for row in forecast_rows}), title="Forecast horizon (days)"),
            y=alt.Y("mae:Q", title="Mean absolute error (lower is better)"),
            color=alt.Color("model:N", title="Model"),
            detail="model:N",
            tooltip=["target:N", "model:N", "horizon_days:O", alt.Tooltip("mae:Q", format=".3f")],
        ).properties(width=320, height=230).facet(
            column=alt.Column("target:N", title="Forecast target")
        )
        display(forecast_chart)
    if winner_rows:
        print("Empirical and statistically supported winners by horizon and target:")
        display(pl.DataFrame(winner_rows).sort("horizon_days", "target"))
    if pairwise_rows:
        print("One-day horizon pairwise results (MAE difference is model A minus model B):")
        pairwise_view = pl.DataFrame(pairwise_rows).select(
            "target", "model_a", "model_b", "median_mae_difference_a_minus_b", "p_value", "p_value_holm"
        )
        display(pairwise_view.sort("target", "p_value_holm"))
else:
    forecast_report = {}
    print(f"No forecast report found at {forecast_path}. Run `uv run health-forecast` first.")

Forecast horizons in report: [1, 3, 7, 14]


alt.FacetChart(...)

Empirical and statistically supported winners by horizon and target:


horizon_days,target,empirical_best,supported_best,subjects
i64,str,str,null,i64
1,"""activity_score""","""lightgbm""",null,20
1,"""recovery_score""","""rolling""",null,20
3,"""activity_score""","""elasticnet""",null,20
3,"""recovery_score""","""rolling""",null,20
7,"""activity_score""","""elasticnet""",null,20
7,"""recovery_score""","""elasticnet""",null,20
14,"""activity_score""","""elasticnet""",null,20
14,"""recovery_score""","""elasticnet""",null,20


One-day horizon pairwise results (MAE difference is model A minus model B):


target,model_a,model_b,median_mae_difference_a_minus_b,p_value,p_value_holm
str,str,str,f64,f64,f64
"""activity_score""","""persistence""","""rolling""",0.168811,0.000063,0.000944
"""activity_score""","""seasonal_persistence""","""rolling""",0.191549,0.000168,0.00235
"""activity_score""","""seasonal_persistence""","""lightgbm""",0.167664,0.00021,0.002728
"""activity_score""","""seasonal_persistence""","""elasticnet""",0.163169,0.000261,0.003136
"""activity_score""","""seasonal_persistence""","""ridge""",0.165858,0.000483,0.005308
…,…,…,…,…,…
"""recovery_score""","""lightgbm""","""elasticnet""",0.013981,0.039989,0.359905
"""recovery_score""","""rolling""","""ridge""",-0.002642,0.595819,1.0
"""recovery_score""","""rolling""","""lightgbm""",0.009081,0.756166,1.0


## 6. Visualizing in the dashboard

The React dashboard uses real data exported from `data/model/daily_features.parquet`. Run `uv run health-export` to write `dashboard/public/data/index.json` and one `subjects/<subject_id>.json` file per subject. Vite serves these files under `/data/`: `src/data/source.ts` fetches `index.json` first for the subject list and model settings, then fetches the selected subject JSON; `App.tsx` supplies that data to the views. The export contains observed daily metric series, positions, velocities, cluster summaries, factor contributions, and forecast paths. What-if controls remain browser-side simulations based on the selected subject's exported data.

Build current features and exports with `uv run health-model && uv run health-export`, then start the UI from `dashboard/` with `npm install` and `npm run dev`. The diagram traces the real artifact-to-browser loading path. See [dashboard/README.md](../dashboard/README.md), [App.tsx](../dashboard/src/App.tsx), [source.ts](../dashboard/src/data/source.ts), and [export_cli.py](../src/health/export_cli.py).

In [12]:
dashboard_steps = [
    {"step": "01 · Features", "label": "daily_features.parquet", "kind": "Model artifact", "detail": "Real daily features produced by health-model"},
    {"step": "02 · Export", "label": "health-export", "kind": "Export", "detail": "Build static dashboard JSON from daily features"},
    {"step": "03 · Files", "label": "index.json + subjects/*.json", "kind": "Static JSON", "detail": "Index plus one real-data payload per subject"},
    {"step": "04 · Fetch", "label": "loadIndex → loadSubject", "kind": "Browser", "detail": "Load the index, then the selected subject payload"},
    {"step": "05 · Views", "label": "Aggregate + Daily", "kind": "Dashboard", "detail": "Render exported history, metrics, and forecasts"},
]
dashboard_order = [row["step"] for row in dashboard_steps]
dashboard_blocks = alt.Chart(alt.Data(values=dashboard_steps)).mark_rect(
    cornerRadius=4
).encode(
    x=alt.X("step:O", sort=dashboard_order, axis=None),
    y=alt.value(30),
    color=alt.Color("kind:N", legend=None),
    tooltip=["label:N", "detail:N"],
)
dashboard_labels = alt.Chart(alt.Data(values=dashboard_steps)).mark_text(
    color="white", fontSize=10, fontWeight="bold"
).encode(
    x=alt.X("step:O", sort=dashboard_order, axis=None),
    y=alt.value(53),
    text="label:N",
)
display(
    (dashboard_blocks + dashboard_labels)
    .properties(width=850, height=100, title="Real data path into the dashboard")
)

alt.LayerChart(...)

## Reproduce and read further

Run the pipeline from the repository root:

```sh
uv sync --all-groups
uv run health-prep scan
# Fill any requested local subject/source mappings, then:
uv run health-prep parse
uv run health-model
uv run health-forecast
uv run health-export
```

Open the interactive real-artifact report with `uv run marimo edit --watch --no-token notebooks/embeddings.py`. To run the real-data dashboard, use `cd dashboard`, `npm install`, and `npm run dev` after exporting its JSON files.

Useful entry points: [README.md](../README.md), [metric catalogue](../src/health/metrics.py), [feature decisions and model caveats](../docs/embedding-baseline.md), [forecast implementation](../src/health/forecast.py), and [dashboard guide](../dashboard/README.md). The test suite is run with `uv run pytest`.

### Current boundaries

The repository documents merging, deduplication, and workout processing as unfinished. Dashboard JSON is generated locally from daily features and remains gitignored with the underlying `data/`; rerun `health-export` after rebuilding the model artifacts. The dashboard's what-if paths are simulations, not new observations. Model comparisons use a small number of subjects and should be treated as a baseline, not population-level evidence. Keep all raw exports and local mapping files out of version control.

**Takeaway:** the branch builds a tested path from heterogeneous personal-health exports to validated daily features, exploratory embeddings, leakage-aware forecasts, and a React dashboard backed by locally exported real data.